# Practical 8 — Word2Vec & Word Embeddings

**Course:** NLP
**Name:**  <!-- fill in -->
**Date:**  <!-- fill in -->

## Aim
To train a Word2Vec model on our own review corpus, compare it against pretrained GloVe embeddings trained on a much larger corpus, and test how each handles out-of-vocabulary words - including a brief look at FastText's subword approach as a partial fix.

## Theory

**Word2Vec** learns dense, low-dimensional vectors for words based on the *distributional hypothesis*: words that appear in similar contexts tend to have similar meanings, so words used in similar contexts end up with similar vectors. Two training modes: **Skip-gram** (predict surrounding context words from a target word) and **CBOW** (predict a target word from its context) — gensim's `Word2Vec` supports both via the `sg` parameter.

Unlike TF-IDF/Bag-of-Words (Practical 7), which only capture *exact word matches*, embeddings can capture semantic relationships — similar words end up near each other in vector space even if they never literally co-occur.

**The catch: this only works with a lot of data.** Meaningful embeddings emerge from statistical co-occurrence patterns across millions of words. Our dataset has 15 short reviews and about 136 unique words — nowhere close to enough. This practical is built specifically to make that limitation concrete: training our own tiny model, and then comparing it directly against **GloVe** vectors pretrained on 6 billion tokens of Wikipedia + Gigaword text, on the exact same words.

There's a second issue specific to Word2Vec: it can only produce a vector for a word it saw during training — anything else is completely **out-of-vocabulary (OOV)** with no vector at all. **FastText** addresses this by representing each word as a bag of character n-grams, so it can approximate a vector for an unseen word from the n-grams it does recognize, even if that word never appeared during training.

## Algorithm

1. Train a Word2Vec model on our own corpus (15 reviews) and query `most_similar` for a few words.
2. Download pretrained GloVe vectors (trained on billions of tokens) and run the identical queries.
3. Compare the two sets of results directly - do the custom-trained results look semantically sensible, or essentially random?
4. Test an out-of-vocabulary query against GloVe (a word not in its vocabulary) and observe what happens.
5. Train a small FastText model on our own corpus and test it on a word variant it never saw during training, to see whether it can still produce a vector where Word2Vec could not.

In [2]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.4/24.4 MB 4.4 MB/s  0:00:05m0:00:0100:01

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [1]:
import sys, os
sys.path.append(os.path.abspath("../python"))

import pandas as pd

import preprocessing
import tokenizer
import embeddings

df = pd.read_csv("../datasets/sample_reviews.csv")
df["tokens"] = df["review"].apply(lambda r: tokenizer.regex_word_tokenize(preprocessing.clean_text(r)))
token_lists = df["tokens"].tolist()

print(f"Loaded {len(df)} reviews, {sum(len(t) for t in token_lists)} total tokens")


Loaded 15 reviews, 194 total tokens


### Step 1 — Train Word2Vec on our own corpus

In [2]:
w2v_model = embeddings.train_word2vec(token_lists)

test_words = ["movie", "good", "bad"]
for word in test_words:
    result = embeddings.safe_most_similar(w2v_model, word, topn=5)
    print(f"Most similar to '{word}' (OUR MODEL): {result}")
    print()


Most similar to 'movie' (OUR MODEL): [('are', 0.30424800515174866), ('was', 0.2941642701625824), ('period', 0.27832356095314026), ('far', 0.26817241311073303), ('wrong', 0.24706274271011353)]

Most similar to 'good' (OUR MODEL): [('before', 0.3913078308105469), ('but', 0.3222843408584595), ('decade', 0.2793152928352356), ('stars', 0.2590988576412201), ('dragged', 0.2585451900959015)]

Most similar to 'bad' (OUR MODEL): [('looked', 0.3233848214149475), ('waste', 0.29469525814056396), ('wow', 0.2908223271369934), ('robert', 0.28880128264427185), ('again', 0.2842925190925598)]



**Look closely: do these "similar" words actually seem semantically related to each query word, or do they look close to random? Keep this in mind for Step 2.**

### Step 2 — Load pretrained GloVe and run the same queries

This downloads a ~66MB model the first time (cached afterward under `~/gensim-data`) - requires an internet connection and may take a minute on first run.

In [3]:
glove = embeddings.load_pretrained_glove("glove-wiki-gigaword-50")

for word in test_words:
    result = embeddings.safe_most_similar(glove, word, topn=5)
    print(f"Most similar to '{word}' (GLOVE, 6B tokens): {result}")
    print()


[==================================================] 100.0% 66.0/66.0MB downloaded
Most similar to 'movie' (GLOVE, 6B tokens): [('movies', 0.9322481155395508), ('film', 0.9310100078582764), ('films', 0.8937394022941589), ('comedy', 0.8902586698532104), ('hollywood', 0.8718216419219971)]

Most similar to 'good' (GLOVE, 6B tokens): [('better', 0.9284391403198242), ('really', 0.9220624566078186), ('always', 0.9165270924568176), ('sure', 0.903351366519928), ('something', 0.9014207720756531)]

Most similar to 'bad' (GLOVE, 6B tokens): [('worse', 0.8878378868103027), ('unfortunately', 0.8650501370429993), ('too', 0.8608258366584778), ('really', 0.8486316204071045), ('little', 0.842767059803009)]



### Step 3 — Out-of-vocabulary test against GloVe

In [4]:
# "wasnt" only exists in this mangled form because Practical 1's clean_text()
# strips apostrophes before tokenization - test whether GloVe's huge vocabulary
# includes this specific mangled form or not.
oov_test_word = "wasnt"
result = embeddings.safe_most_similar(glove, oov_test_word, topn=5)
print(f"GloVe result for '{oov_test_word}': {result}")


GloVe result for 'wasnt': [('realy', 0.8441590666770935), ('unbelieveable', 0.7592869997024536), ('kloot', 0.7482977509498596), ('eveything', 0.7441456913948059), ('couldnt', 0.7380183935165405)]


### Step 4 — FastText: can it handle a word it never saw?

In [5]:
ft_model = embeddings.train_fasttext(token_lists)

# Pick a real word from the corpus, and a made-up near-miss spelling of it
# that was NOT in the training data.
known_word = "boredom"  # confirmed present in review 7 - adjust if you change the dataset
unseen_variant = "boredomm"  # deliberately misspelled, should not appear in training data

print(f"Is '{known_word}' in the FastText vocabulary?  {known_word in ft_model.wv.key_to_index}")
print(f"Is '{unseen_variant}' in the FastText vocabulary? {unseen_variant in ft_model.wv.key_to_index}")
print()

try:
    vector = ft_model.wv[unseen_variant]
    print(f"FastText produced a vector for the unseen '{unseen_variant}' anyway (first 5 dims): {vector[:5]}")
except KeyError:
    print(f"FastText could NOT produce a vector for '{unseen_variant}'")

try:
    w2v_vector = w2v_model.wv[unseen_variant]
    print(f"Word2Vec produced a vector for '{unseen_variant}' (first 5 dims): {w2v_vector[:5]}")
except KeyError:
    print(f"Word2Vec could NOT produce a vector for '{unseen_variant}' (expected - confirm this is what happened)")


Is 'boredom' in the FastText vocabulary?  True
Is 'boredomm' in the FastText vocabulary? False

FastText produced a vector for the unseen 'boredomm' anyway (first 5 dims): [ 0.00065201  0.00021114  0.00099324 -0.00043614 -0.00507787]
Word2Vec could NOT produce a vector for 'boredomm' (expected - confirm this is what happened)


**Note: if "boring" isn't actually a word in your dataset, swap `known_word` for one that is - check the dataset/tokens first rather than assuming.**

## Observations & Conclusion

Answer these based on what you actually saw when you ran the notebook:

- Did the custom Word2Vec model's "most similar" results for "movie", "good", "bad" look semantically meaningful, or essentially arbitrary? What does that tell you about the 15-document, ~136-word corpus relative to what Word2Vec actually needs?
- Did GloVe's results for the same three words look noticeably more sensible? Give a specific example of a GloVe result you'd consider a genuinely good match.
- Did "wasnt" turn out to be in GloVe's vocabulary or not? Either way, what does that say about training on real-world (sometimes messy) text at massive scale, versus training on our small, apostrophe-stripped corpus?
- Did FastText succeed where Word2Vec failed on the unseen word variant? Explain *why*, in terms of what FastText actually does differently (character n-grams) rather than just restating that it worked.

### Answer:
- The custom Word2Vec model produced word similarities based on the small movie review dataset, but many of the nearest neighbors, such as "movie" → "are" and "good" → "before", were not semantically meaningful because the corpus was too small to learn rich word relationships. In contrast, the pre-trained GloVe model returned much more intuitive results, such as "movie" → "film", "movies", "films" and "bad" → "worse", reflecting knowledge learned from billions of tokens. Interestingly, GloVe successfully recognized the misspelled word "wasnt" and returned semantically related neighbors instead of treating it as an out-of-vocabulary (OOV) term. This suggests that, because GloVe was trained on a massive corpus of 6 billion tokens, it has encountered common informal spellings and typographical variations such as "wasnt", "couldnt", and "realy" often enough to learn meaningful vector representations for them. Rather than illustrating a limitation of OOV handling, this result demonstrates the robustness of large pre-trained embedding models to frequently occurring non-standard spellings. The comparison between Word2Vec and FastText demonstrated a genuine OOV case instead: FastText generated a vector for the truly unseen "boredomm" using subword information, whereas Word2Vec could not produce any embedding for it at all. Overall, the experiment highlights that pre-trained embeddings generally provide higher-quality semantic representations than models trained on very small datasets, while FastText offers a real advantage over Word2Vec specifically for genuinely novel or misspelled words outside a small training vocabulary.



---
## Viva Prep — Practice Questions

1. **What is the distributional hypothesis, and how does Word2Vec use it?**
   The idea that words appearing in similar contexts tend to have similar meanings. Word2Vec uses this by training a model to predict a word from its context (or vice versa), which causes words used in similar contexts to end up with similar vectors as a side effect of that prediction task.

2. **Why did our custom-trained Word2Vec model likely produce poor-quality similarity results?**
   Word2Vec needs large amounts of text to reliably learn co-occurrence patterns; 15 short reviews (~136 unique words) provide far too little data for the model to learn anything beyond noise.

3. **What's the difference between Skip-gram and CBOW?**
   Skip-gram predicts surrounding context words given a target word; CBOW predicts a target word given its surrounding context. They're roughly inverse formulations of the same underlying idea.

4. **Why can't Word2Vec produce a vector for an out-of-vocabulary word, and how does FastText solve this?**
   Word2Vec only stores a vector per whole word it saw during training - an unseen word has no entry at all. FastText instead represents each word as a combination of character n-gram vectors, so it can approximate a vector for a new word from the n-grams it recognizes, even without having seen that exact word before.

5. **What's the practical trade-off of using pretrained embeddings (like GloVe) instead of training your own on a small, domain-specific corpus?**
   Pretrained embeddings capture much richer, more reliable semantic relationships thanks to massive training data, but they're trained on general text and may not capture domain-specific usage or vocabulary; training your own is domain-specific but requires far more data than most individual projects (like this one) actually have.
